In [1]:
!pip install transformers
!pip install PyMuPDF
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 63.1 MB/s eta 0:00:00


In [2]:
import fitz  # PyMuPDF
from transformers import pipeline

In [36]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Uploaded file:", pdf_path)

Saving Test PDF (Hard).pdf to Test PDF (Hard).pdf
Uploaded file: Test PDF (Hard).pdf


In [37]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""

    for page in doc:
        text += page.get_text()

    return text

pdf_text = extract_text_from_pdf(pdf_path)

print(pdf_text[:1000])  # preview

Bangladesh(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&
&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&
&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%
%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%
%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$
$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((
((((((_. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and 
vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the country’s development. 
 
One of the most important natural features of Bangladesh is the 
(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&()
)))))))))(((((((((((((((_$$$$$$$$$$$$$$$

In [38]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text

pdf_text = clean_text(pdf_text)

In [39]:
def chunk_text(text, chunk_size=400):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))

    return chunks

chunks = chunk_text(pdf_text)

print("Total chunks:", len(chunks))

Total chunks: 2


Note: This system uses a BERT-based model fine-tuned on SQuAD-style question answering (extractive QA). It is designed to extract exact answers from the given text. Questions that require explanatory or descriptive answers may not work well. For generating explanatory answers, GPT-style or LLaMA-type models are more suitable.

In [40]:
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/xlm-roberta-large-squad2",
    tokenizer="deepset/xlm-roberta-large-squad2"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForQuestionAnswering LOAD REPORT from: deepset/xlm-roberta-large-squad2
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [43]:
import re

def split_questions(text):
    questions = re.split(r'\?\s*', text)
    questions = [q.strip() + '?' for q in questions if q.strip()]
    return questions

In [41]:
def get_answer(question, chunks):
    best_answer = ""
    best_score = 0

    for chunk in chunks:
        result = qa_pipeline(question=question, context=chunk)

        if result['score'] > best_score:
            best_score = result['score']
            best_answer = result['answer']

    # Low confidence handling
    if best_score < 0.2:
        return (
            "⚠️ The model is not confident about this answer.\n"
            "It may be incorrect or not clearly available in the PDF.\n"
            "Also note: this system is extractive and may not handle explanatory questions well.",
            best_score
        )

    # Medium confidence (optional but useful)
    elif best_score < 0.4:
        return (
            f"⚠️ Possible answer (low confidence): {best_answer}\n"
            "This answer might be incomplete or inaccurate.",
            best_score
        )

    return best_answer, best_score

In [49]:
while True:
    user_input = input("\nAsk a question (or type 'exit'): ")

    if user_input.lower() == "exit":
        break

    # print("\n" + "="*40)

    questions = split_questions(user_input)

    if len(questions) > 1:
        print("⚠️ Multiple questions detected. Answering individually.\n")

        for i, question in enumerate(questions, 1):
            answer, score = get_answer(question, chunks)

            print("Answer:", answer)
            print("Confidence score:", score)
            print("-"*40)  # divider between questions

    else:
        answer, score = get_answer(questions[0], chunks)
        print("Answer:", answer)
        print("Confidence score:", score)

    print("="*40 + "\n")  # end of one input block


Ask a question (or type 'exit'): What is the capital city of Bangladesh?
Answer: Dhaka.
Confidence score: 0.970827853307128


Ask a question (or type 'exit'): Which river is considered the lifeline of Bangladesh?
Answer: Padma River,
Confidence score: 1.6225187091331463


Ask a question (or type 'exit'): In which year did Bangladesh gain independence?
Answer: 1971.
Confidence score: 0.9962125490419567


Ask a question (or type 'exit'): What is the official language of Bangladesh?
Answer: Bengali,
Confidence score: 0.772847268730402


Ask a question (or type 'exit'): What is the name of the world’s largest mangrove forest located in Bangladesh?
Answer: Sundarbans,
Confidence score: 0.853174354667317


Ask a question (or type 'exit'): Which currency is used in Bangladesh? What is the national animal of Bangladesh?
⚠️ Multiple questions detected. Answering individually.

Answer: Bangladeshi Taka.
Confidence score: 0.9224153244576883
----------------------------------------
Answer: Royal 